# Telemetry Log Parser and Detection Visualizer

This notebook parses telemetry log files and visualizes detections on images.

In [ ]:
import re
import os
from datetime import datetime
from typing import List, Dict, Tuple
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Configuration

In [ ]:
# Configuration
LOG_FILE = 'pg2gw_fp.log'  # Change this to your log file name
IMGS_FOLDER = 'imgs'
CURRENT_DIR = Path('.')

## Parse Log File

In [ ]:
class Detection:
    """Represents a single detection"""
    def __init__(self, pixel_x, pixel_y, cm_x, cm_y, conf, box_x, box_y, box_w, box_h, sensor_id):
        self.pixel_x = pixel_x
        self.pixel_y = pixel_y
        self.cm_x = cm_x
        self.cm_y = cm_y
        self.confidence = conf
        self.box_x = box_x
        self.box_y = box_y
        self.box_w = box_w
        self.box_h = box_h
        self.sensor_id = sensor_id
    
    def __repr__(self):
        return (f"Detection(pixel=({self.pixel_x},{self.pixel_y}), "
                f"box=({self.box_x},{self.box_y},{self.box_w},{self.box_h}), "
                f"conf={self.confidence:.3f})")

class TelemetryMessage:
    """Represents a telemetry message with detections"""
    def __init__(self, timestamp, sensor_id, length):
        self.timestamp = timestamp
        self.sensor_id = sensor_id
        self.length = length
        self.detections = []
    
    def add_detection(self, detection):
        self.detections.append(detection)
    
    def __repr__(self):
        return (f"TelemetryMessage(timestamp={self.timestamp}, "
                f"sensor={self.sensor_id}, "
                f"detections={len(self.detections)})")

In [ ]:
def parse_log_file(log_file_path: str) -> List[TelemetryMessage]:
    """
    Parse the log file and extract telemetry messages with their detections.
    
    Args:
        log_file_path: Path to the log file
        
    Returns:
        List of TelemetryMessage objects
    """
    # Regular expressions for parsing
    telemetry_pattern = re.compile(
        r'(\w{3}\s+\d{2}\s+\d{2}:\d{2}:\d{2}).*Received telemetry data, length: (\d+), from thing ([\w\-]+)'
    )
    
    detection_pattern = re.compile(
        r'(\w{3}\s+\d{2}\s+\d{2}:\d{2}:\d{2}).*Detection translated for sensor ([\w\-]+): '
        r'pixel\((\d+),(\d+)\) -> cm\((-?\d+),(-?\d+)\). '
        r'conf = ([\d.]+); box: x = (\d+), y = (\d+), w = (\d+), h = (\d+)'
    )
    
    telemetry_messages = []
    current_message = None
    
    with open(log_file_path, 'r') as f:
        for line in f:
            # Check for telemetry message
            telemetry_match = telemetry_pattern.search(line)
            if telemetry_match:
                # Save previous message if it exists
                if current_message is not None:
                    telemetry_messages.append(current_message)
                
                # Create new message
                timestamp = telemetry_match.group(1)
                length = int(telemetry_match.group(2))
                sensor_id = telemetry_match.group(3)
                current_message = TelemetryMessage(timestamp, sensor_id, length)
                continue
            
            # Check for detection
            detection_match = detection_pattern.search(line)
            if detection_match and current_message is not None:
                timestamp = detection_match.group(1)
                sensor_id = detection_match.group(2)
                pixel_x = int(detection_match.group(3))
                pixel_y = int(detection_match.group(4))
                cm_x = int(detection_match.group(5))
                cm_y = int(detection_match.group(6))
                conf = float(detection_match.group(7))
                box_x = int(detection_match.group(8))
                box_y = int(detection_match.group(9))
                box_w = int(detection_match.group(10))
                box_h = int(detection_match.group(11))
                
                detection = Detection(pixel_x, pixel_y, cm_x, cm_y, conf, 
                                    box_x, box_y, box_w, box_h, sensor_id)
                current_message.add_detection(detection)
    
    # Don't forget the last message
    if current_message is not None:
        telemetry_messages.append(current_message)
    
    return telemetry_messages

In [ ]:
# Parse the log file
log_path = CURRENT_DIR / LOG_FILE
telemetry_messages = parse_log_file(log_path)

print(f"Parsed {len(telemetry_messages)} telemetry messages")
print(f"\nFirst 5 messages:")
for i, msg in enumerate(telemetry_messages[:5]):
    print(f"{i+1}. {msg}")
    for det in msg.detections:
        print(f"   - {det}")

In [ ]:
# Get messages with detections only
messages_with_detections = [msg for msg in telemetry_messages if len(msg.detections) > 0]

print(f"Total messages: {len(telemetry_messages)}")
print(f"Messages with detections: {len(messages_with_detections)}")
print(f"Messages without detections: {len(telemetry_messages) - len(messages_with_detections)}")

# Statistics
total_detections = sum(len(msg.detections) for msg in telemetry_messages)
print(f"\nTotal detections: {total_detections}")
if messages_with_detections:
    avg_detections = total_detections / len(messages_with_detections)
    print(f"Average detections per message (with detections): {avg_detections:.2f}")

## Visualization Methods

In [ ]:
def draw_detections_on_image(image_path: str, detections: List[Detection], 
                            show_confidence: bool = True,
                            show_pixel_coords: bool = False,
                            show_cm_coords: bool = False,
                            box_color: Tuple[int, int, int] = (0, 255, 0),
                            thickness: int = 2) -> np.ndarray:
    """
    Draw detections on an image.
    
    Args:
        image_path: Path to the image file
        detections: List of Detection objects
        show_confidence: Whether to show confidence scores
        show_pixel_coords: Whether to show pixel coordinates
        show_cm_coords: Whether to show cm coordinates
        box_color: Color of bounding boxes (B, G, R)
        thickness: Thickness of bounding box lines
        
    Returns:
        Image with drawn detections
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    
    # Draw each detection
    for det in detections:
        # Draw bounding box
        x1, y1 = det.box_x, det.box_y
        x2, y2 = det.box_x + det.box_w, det.box_y + det.box_h
        cv2.rectangle(img, (x1, y1), (x2, y2), box_color, thickness)
        
        # Draw center point
        center_x = det.pixel_x
        center_y = det.pixel_y
        cv2.circle(img, (center_x, center_y), 3, (0, 0, 255), -1)
        
        # Prepare text
        text_parts = []
        if show_confidence:
            text_parts.append(f"{det.confidence:.2f}")
        if show_pixel_coords:
            text_parts.append(f"px: ({det.pixel_x},{det.pixel_y})")
        if show_cm_coords:
            text_parts.append(f"cm: ({det.cm_x},{det.cm_y})")
        
        # Draw text
        if text_parts:
            text = " | ".join(text_parts)
            # Position text above the box
            text_y = max(y1 - 10, 20)
            cv2.putText(img, text, (x1, text_y), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, box_color, 1, cv2.LINE_AA)
    
    return img


def find_image_by_name(imgs_folder: str, image_name: str) -> str:
    """
    Find image file in the imgs folder by name.
    Supports various image extensions.
    
    Args:
        imgs_folder: Path to the images folder
        image_name: Name of the image (with or without extension)
        
    Returns:
        Full path to the image file
    """
    imgs_path = Path(imgs_folder)
    
    # Common image extensions
    extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
    
    # Try with the provided name first
    candidate = imgs_path / image_name
    if candidate.exists():
        return str(candidate)
    
    # Try adding extensions
    base_name = Path(image_name).stem
    for ext in extensions:
        candidate = imgs_path / f"{base_name}{ext}"
        if candidate.exists():
            return str(candidate)
    
    raise FileNotFoundError(f"Image not found: {image_name} in {imgs_folder}")


def display_image_with_detections(image_path: str, detections: List[Detection],
                                 figsize: Tuple[int, int] = (12, 8),
                                 **draw_kwargs):
    """
    Display image with detections using matplotlib.
    
    Args:
        image_path: Path to the image file
        detections: List of Detection objects
        figsize: Figure size for matplotlib
        **draw_kwargs: Additional arguments for draw_detections_on_image
    """
    img = draw_detections_on_image(image_path, detections, **draw_kwargs)
    
    # Convert BGR to RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"Detections: {len(detections)}")
    plt.tight_layout()
    plt.show()


def save_image_with_detections(image_path: str, detections: List[Detection],
                              output_path: str, **draw_kwargs):
    """
    Save image with detections to file.
    
    Args:
        image_path: Path to the input image file
        detections: List of Detection objects
        output_path: Path to save the output image
        **draw_kwargs: Additional arguments for draw_detections_on_image
    """
    img = draw_detections_on_image(image_path, detections, **draw_kwargs)
    cv2.imwrite(output_path, img)
    print(f"Saved image with {len(detections)} detections to: {output_path}")

## Example Usage

In [ ]:
# Example: Visualize detections from a specific telemetry message
# Choose a message with detections
if messages_with_detections:
    example_message = messages_with_detections[0]
    print(f"Example message: {example_message}")
    print(f"Detections: {len(example_message.detections)}")
    for i, det in enumerate(example_message.detections):
        print(f"  {i+1}. {det}")

In [ ]:
# Example: Draw detections on an image
# Uncomment and modify the image name to match your actual image files

image_name = '6000000-2000-000037-2550'  # Change this to your image name
try:
    image_path = find_image_by_name(IMGS_FOLDER, image_name)
    
    # Get detections from the first message with detections
    if messages_with_detections:
        detections = messages_with_detections[5].detections
        
        # Display the image with detections
        display_image_with_detections(
            image_path, 
            detections,
            show_confidence=True,
            show_pixel_coords=False,
            show_cm_coords=False,
            box_color=(0, 255, 0),  # Green boxes
            thickness=2
        )
except FileNotFoundError as e:
    print(e)

In [ ]:
# Example: Process multiple messages and save annotated images
# Uncomment to use

# output_folder = CURRENT_DIR / 'annotated_images'
# output_folder.mkdir(exist_ok=True)
# 
# for i, msg in enumerate(messages_with_detections[:10]):  # Process first 10 messages
#     try:
#         # Assuming image names correspond to message indices or timestamps
#         # Modify this logic based on your actual image naming convention
#         image_name = f'image_{i}.jpg'  # Adjust naming as needed
#         image_path = find_image_by_name(IMGS_FOLDER, image_name)
#         
#         output_path = output_folder / f'annotated_{i}.jpg'
#         save_image_with_detections(
#             image_path,
#             msg.detections,
#             str(output_path),
#             show_confidence=True,
#             show_pixel_coords=False
#         )
#     except FileNotFoundError:
#         print(f"Image not found for message {i}")
#         continue

## Export Data

In [ ]:
# Convert to list of lists format as requested
# Each element is a list of detections for a telemetry message
detections_list = [msg.detections for msg in telemetry_messages]

print(f"Created list of {len(detections_list)} telemetry messages")
print(f"Messages with detections: {sum(1 for dets in detections_list if len(dets) > 0)}")

In [ ]:
# Access detections by message index
# Example: Get detections from the 5th message
if len(detections_list) > 4:
    message_5_detections = detections_list[4]
    print(f"Message 5 has {len(message_5_detections)} detections:")
    for det in message_5_detections:
        print(f"  - {det}")

In [ ]:
[len(i.detections) for i in messages_with_detections]

In [ ]:
np.mean([len(i.detections) for i in messages_with_detections])